# Lab 13: Interactive Dashboard and Deployment
This notebook demonstrates the Movie Industry Analytics / Real Estate Market Monitor interactive dashboard, MongoDB database seeding, and containerized deployment workflow.

## 1. MongoDB Seeding
Run the database seeding script to insert cleaned listings into MongoDB and create indexes.

In [2]:
%run ../scripts/seed_mongo.py

Seeding MongoDB...
Seeding Complete: Inserted 246 records.
Indexes created on 'type', 'year', and 'title'.


## 2. Test Data Access
Test that `src/dashboard/data_access.py` successfully reads from MongoDB and falls back to CSV if needed.

In [3]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.dashboard.data_access import load_data, get_available_genres, get_year_range

df = load_data()
print("✅ Data Loaded successfully!")
print(f"   - Total Records: {len(df)}")
print(f"   - Columns: {list(df.columns)}")
print(f"   - Property Types: {get_available_genres(df)}")
print(f"   - Year Range: {get_year_range(df)}")

c:\Users\User\Desktop\Real-Estate-Market-Monitor\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


✅ Data Loaded successfully!
   - Total Records: 246
   - Columns: ['price', 'collected_at', 'source', 'listing_id', 'coordinates', 'area', 'type', 'description', 'title', 'extracted_price_mentions', 'short_description_flag', 'year']
   - Property Types: ['Industrial', 'Multi-Family', 'Office', 'Retail']
   - Year Range: (2026, 2026)


## 3. Test Dashboard Layout and Callbacks Registration
Instantiate the Dash app object and load layout and callbacks to ensure no import errors exist.

In [4]:
import dash
import dash_bootstrap_components as dbc
from src.dashboard.layout import get_layout
from src.dashboard.callbacks import register_callbacks

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.CYBORG])
app.layout = get_layout(df)
register_callbacks(app)
print("✅ Dashboard layout and callbacks initialized successfully without errors!")

✅ Dashboard layout and callbacks initialized successfully without errors!


## 4. Deployment Documentation
### Prerequisites
- **Docker Desktop**: Installed and running on your host system (Windows, macOS, or Linux).
- **Ports**: Ensure ports `8050` (Dash dashboard) and `27017` (MongoDB) are available and not occupied by other services on your host.
- **Python Environment**: Python 3.12 with dependencies installed if testing locally.

### Step-by-Step Deployment Commands
1. **Build and Start the Container Stack** (starts MongoDB and the Dash dashboard in background detached mode):
   ```bash
   docker compose up --build -d
   ```

2. **Seed the database from the Host machine** (this will populate MongoDB inside the docker network):
   ```bash
   python scripts/seed_mongo.py
   ```
   *(Alternatively, run the seeding script inside the container using `docker exec -it dash_dashboard python scripts/seed_mongo.py`)*

### Verifying the Deployment
- Open your web browser and navigate to: **[http://localhost:8050](http://localhost:8050)**.
- **What to Look For**:
  - Sleek, dark **Cyborg** theme with the title **"REAL ESTATE MARKET MONITOR"**.
  - **KPI Cards** showing updated figures: Total Listings (246), average price, average area, and average price/sqft.
  - **Interactive dropdowns and sliders** to filter listings dynamically.
  - **4 Plotly Charts** updating automatically based on filters: Price Bar Chart, Property Type Composition Pie Chart, Price Distribution Violin Plot, and Property Area Histogram.
  - **Simulated Real-Time Price Stream** at the bottom, updating with new data point markers every 3 seconds.

### Stopping the Stack Cleanly
- To stop and remove the container stack while preserving the database data, run:
  ```bash
  docker compose down
  ```
- To stop, remove containers, and completely wipe the MongoDB volume data to start fresh:
  ```bash
  docker compose down -v
  ```

### Troubleshooting Tips
- **Port 8050 is already in use**:
  - Run `docker ps` to see if a previous dashboard container is still running, and stop it using `docker stop dash_dashboard`.
  - On Windows, if port 8050 is occupied by another local program, open `docker-compose.yml` and modify the web ports from `"8050:8050"` to a different host port like `"8080:8050"`, then rerun `docker compose up --build -d`. You will then access the app at `http://localhost:8080`.
- **MongoDB Connection Refused**:
  - Make sure the MongoDB container is fully initialized. The `depends_on` rule ensures that MongoDB starts before the dashboard, but MongoDB might take a few seconds to accept connections. If the dashboard crashes initially, Docker will automatically restart it according to the `restart: always` policy.